# ¿El filtro de idioma nos arruina los textos? (subtítulos)

OpenSubtitles trae cada línea en español **y** en catalán, lo que nos da la "respuesta correcta" gratis:
- lado `es` → el filtro debería **aceptarlo**
- lado `ca` → el filtro debería **rechazarlo**

## Por qué esta prueba replica el pipeline real

El filtro **no se aplica a cualquier línea**. Se aplica solo en dos lugares, con condiciones previas. La prueba usa esas mismas condiciones y constantes (con su fuente):

| Qué | Condición real | Fuente |
|---|---|---|
| Textos "estándar" | solo si `len(texto) > min_len`, y recién ahí `es_espanol(texto)` | `src/datos/carga.py` líneas 77-78 |
| `min_len` en subtítulos | 20 caracteres (valor por defecto) | `carga.py` línea 123 |
| Preguntas | `extraer_preguntas()` parte por `.` o salto de línea y exige `¿` y `?` | `carga.py` líneas 30-32 |
| Largo mínimo de pregunta | 2 palabras en subtítulos | `codigoTP.ipynb` línea 690 (`min_palabras_pregunta=2`) |
| Filtro sobre cada pregunta | `es_espanol(pregunta)` | `carga.py` línea 85 |

Una versión anterior de esta prueba filtraba **todas** las líneas, incluso "Hola" o "Jesús.", que el pipeline nunca manda a `es_espanol`. Eso inflaba el porcentaje de rechazo.

**Diferencias que quedan (aproximaciones):**
- En el pipeline, las preguntas se minan solo de las filas que *no* entraron como estándar. Acá se minan de todas las filas.
- La muestra es de 3000 filas con otro `buffer_size` de mezcla (el pipeline usa 10.000), así que no son exactamente las mismas filas.

In [18]:
# %pip install datasets langdetect pandas   # descomentar si falta algo
import sys, os
sys.path.insert(0, os.getcwd())               # correr desde la raíz del repo
import pandas as pd
from datasets import load_dataset
from langdetect import detect_langs
from src.datos.carga import es_espanol, extraer_preguntas   # el filtro EXACTO del pipeline

MIN_LEN = 20            # cargar_opensubtitles(min_len=20)        -> carga.py:123
MIN_PAL_PREGUNTA = 2    # cargar_opensubtitles(min_palabras_pregunta=2) -> codigoTP.ipynb:690

ds = load_dataset("homersimpson/opensubtitles_es", split="train", streaming=True).shuffle(seed=101, buffer_size=2000)
filas = list(ds.take(3000))
es = [f["translation"]["es"].strip() for f in filas]
ca = [f["translation"]["ca"].strip() for f in filas]
print(len(es), "filas es/ca")

3000 filas es/ca


## 1. Textos estándar: solo líneas de más de `MIN_LEN` caracteres

Es el caso `len(texto) > min_len` → `es_espanol(texto)` (carga.py:77-78).

In [19]:
std = pd.DataFrame({"es": es, "ca": ca})
std = std[std["es"].str.len() > MIN_LEN].copy()      # condición real del pipeline
std["es_ok"] = std["es"].map(es_espanol)             # True = el filtro lo deja pasar
std["ca_ok"] = std["ca"].map(es_espanol)

N = len(std)
rech = (~std["es_ok"]).sum()
print(f"Líneas que llegan al filtro (>{MIN_LEN} caracteres): {N}")
print(f"Español rechazado: {rech} de {N} ({rech/N:.1%})   <- ideal: 0")
print(f"Español aceptado:  {N - rech} de {N} ({1 - rech/N:.1%})")

dif = std[std["es"].str.lower() != std["ca"].str.lower()]   # solo donde es y ca realmente difieren
print(f"Catalán que se cuela: {dif['ca_ok'].sum()} de {len(dif)} ({dif['ca_ok'].mean():.1%})   <- ideal: 0")

Líneas que llegan al filtro (>20 caracteres): 1895
Español rechazado: 261 de 1895 (13.8%)   <- ideal: 0
Español aceptado:  1634 de 1895 (86.2%)
Catalán que se cuela: 57 de 1893 (3.0%)   <- ideal: 0


## 2. Preguntas `¿...?` minadas con `extraer_preguntas` (carga.py:30-32, 85)

In [20]:
preg = [q for t in es for q in extraer_preguntas(t, min_palabras=MIN_PAL_PREGUNTA)]
preg_ok = [es_espanol(q) for q in preg]
rech_p = preg_ok.count(False)
print(f"Preguntas extraídas del lado español: {len(preg)}")
print(f"Rechazadas: {rech_p} de {len(preg)} ({rech_p/max(len(preg),1):.1%})   <- ideal: 0")

preg_ca = [q for t in ca for q in extraer_preguntas(t, min_palabras=MIN_PAL_PREGUNTA)]
print(f"Preguntas '¿...?' que salen del lado catalán: {len(preg_ca)}   (si es ~0, la extracción ya descarta el catalán)")

Preguntas extraídas del lado español: 500
Rechazadas: 163 de 500 (32.6%)   <- ideal: 0
Preguntas '¿...?' que salen del lado catalán: 17   (si es ~0, la extracción ya descarta el catalán)


## 3. ¿Dónde se concentra lo que se pierde? (textos estándar, por cantidad de palabras)

In [21]:
std["palabras"] = std["es"].str.split().str.len()
std["largo"] = pd.cut(std["palabras"], [0, 3, 5, 8, 12, 1000], labels=["1-3", "4-5", "6-8", "9-12", "13+"])
t = std.groupby("largo", observed=True)["es_ok"].agg(n="size", aceptados="sum")
t["rechazados"] = t["n"] - t["aceptados"]
t["% rechazo"] = (100 * t["rechazados"] / t["n"]).round(1)
t["% del total rechazado"] = (100 * t["rechazados"] / t["rechazados"].sum()).round(1)
t

,n,aceptados,rechazados,% rechazo,% del total rechazado
largo,,,,,
1-3,70,53,17,24.3,6.5
4-5,546,411,135,24.7,51.7
6-8,676,587,89,13.2,34.1
9-12,378,360,18,4.8,6.9
13+,225,223,2,0.9,0.8


## 4. Ejemplos de español que el filtro tira (y qué idioma cree que son)

In [22]:
def idioma(t):
    try: return detect_langs(t)[0].lang
    except Exception: return "?"

malos = std[~std["es_ok"]].head(12)
print("Textos estándar rechazados:")
display(pd.DataFrame({"texto": malos["es"], "langdetect dijo": malos["es"].map(idioma)}))

malas_q = [q for q, ok in zip(preg, preg_ok) if not ok][:12]
print("Preguntas rechazadas:")
display(pd.DataFrame({"pregunta": malas_q, "langdetect dijo": [idioma(q) for q in malas_q]}))

Textos estándar rechazados:


,texto,langdetect dijo
11,Mi madre es la que...,ca
16,Lo de Balotelli va a ser complicado.,it
18,Oye. Necesito tu moto.,it
67,¿Por qué no me lo dijo nunca?,pt
74,"Tenia dos sabuesos Blue Tick, Sam y Finningan.",id
87,"Por favor, estate quieta.",ca
120,A menos que no lo haga.,pt
129,No tengo ninguna en mi agenda.,id
139,"Helmsman, izquierda, 180.",nl
142,Yo conozco a la gente.,it


Preguntas rechazadas:


,pregunta,langdetect dijo
0,"¿Conde Oetsch?""",de
1,¿Por qué no me lo dijo nunca?,pt
2,¿Qué demonios?,pt
3,¿Qué pasa?,fr
4,"¿En qué andas, John?",fr
5,- ¿Tú crees?,ca
6,¿Por qué?,fr
7,¿Qué pasa al final?,ca
8,¿ella mató a Monroe?,ca
9,"Penn State, ¿verdad?",no


## 5. Comparación de filtros alternativos

Cuatro variantes, todas con `langdetect` pero con distinto criterio de rechazo:

| Variante | Qué hace |
|---|---|
| **A actual** | rechaza si el idioma más probable no es `es` (cualquier idioma, cualquier confianza) |
| **B largo ≥ `MIN_CHARS`** | las líneas de menos de `MIN_CHARS` caracteres **no se filtran** (se aceptan); el resto, como A |
| **C solo contaminantes** | rechaza **solo** si `langdetect` dice que es un idioma de `CONTAMINANTES` **y** con probabilidad ≥ `UMBRAL`. Cualquier otra cosa (turco, indonesio, dudas) se acepta |
| **D = B + C** | las dos reglas juntas |

Se miden sobre los mismos datos de las secciones 1 y 2: español rechazado (queremos que baje) y catalán que pasa (queremos que no suba mucho).

In [23]:
import collections

MIN_CHARS = 40                                  # B y D: no filtramos líneas de menos de 40 caracteres (~7 palabras)
CONTAMINANTES = {"ca", "en", "fr", "pt", "it"}  # C y D: idiomas que realmente podrían colarse
UMBRAL = 0.9                                    # C y D: solo rechazamos si langdetect está >= 90% seguro

_cache = {}
def det(t):
    """(idioma más probable, probabilidad) según langdetect. Cacheado para no repetir."""
    if t not in _cache:
        try:
            l = detect_langs(t)[0]; _cache[t] = (l.lang, l.prob)
        except Exception:
            _cache[t] = ("?", 0.0)              # sin letras, p. ej. "..."
    return _cache[t]

def acepta_A(t): return det(t)[0] == "es"                                   # = es_espanol() actual
def acepta_B(t): return len(t) < MIN_CHARS or acepta_A(t)
def acepta_C(t):
    lang, p = det(t)
    return not (lang in CONTAMINANTES and p >= UMBRAL)
def acepta_D(t): return len(t) < MIN_CHARS or acepta_C(t)

variantes = {"A actual": acepta_A, f"B largo>={MIN_CHARS}": acepta_B,
             f"C solo contaminantes>={UMBRAL}": acepta_C, "D = B + C": acepta_D}

ca_dif = dif["ca"]       # catalán de los pares donde es != ca (sección 1)
filas = []
for nombre, f in variantes.items():
    es_r = sum(not f(t) for t in std["es"])
    q_r  = sum(not f(q) for q in preg)
    ca_p = sum(f(t) for t in ca_dif)
    caq_p = sum(f(q) for q in preg_ca)
    filas.append({
        "variante": nombre,
        "español rechazado (textos)": f"{es_r} de {len(std)} ({es_r/len(std):.1%})",
        "preguntas es rechazadas": f"{q_r} de {len(preg)} ({q_r/len(preg):.1%})",
        "catalán que pasa (textos)": f"{ca_p} de {len(ca_dif)} ({ca_p/len(ca_dif):.1%})",
        "preguntas ca que pasan": f"{caq_p} de {len(preg_ca)}",
    })
pd.DataFrame(filas).set_index("variante")

,español rechazado (textos),preguntas es rechazadas,catalán que pasa (textos),preguntas ca que pasan
variante,,,,
A actual,261 de 1895 (13.8%),163 de 500 (32.6%),57 de 1893 (3.0%),0 de 17
B largo>=40,41 de 1895 (2.2%),6 de 500 (1.2%),1114 de 1893 (58.8%),15 de 17
C solo contaminantes>=0.9,87 de 1895 (4.6%),64 de 500 (12.8%),408 de 1893 (21.6%),7 de 17
D = B + C,11 de 1895 (0.6%),1 de 500 (0.2%),1203 de 1893 (63.5%),16 de 17


### ¿Qué idioma le asigna `langdetect` al catalán que se cuela?

Para cada variante: cuántas líneas en catalán pasan y qué idioma dijo `langdetect` que eran. Si dice `es`, el detector se confundió; si dice otro idioma, pasó por la regla de la variante (línea corta, o confianza baja).

In [24]:
for nombre, f in variantes.items():
    pasan = [t for t in ca_dif if f(t)]
    c = collections.Counter(det(t)[0] for t in pasan)
    print(f"{nombre:34} {len(pasan):4} pasan -> idioma según langdetect: {dict(c.most_common(5))}")

A actual                             57 pasan -> idioma según langdetect: {'es': 57}
B largo>=40                        1114 pasan -> idioma según langdetect: {'ca': 831, 'pt': 58, 'fr': 58, 'es': 57, 'it': 31}
C solo contaminantes>=0.9           408 pasan -> idioma según langdetect: {'ca': 199, 'es': 57, 'fr': 31, 'pt': 29, 'it': 21}
D = B + C                          1203 pasan -> idioma según langdetect: {'ca': 899, 'pt': 65, 'fr': 64, 'es': 57, 'it': 36}


### Mirar los casos de la variante D (a ojo)

Lo que mide la tabla es útil, pero lo importante es **leer** qué entra y qué sigue saliendo.

In [25]:
m = pd.Series([t for t in ca_dif if acepta_D(t)])
m = m.sample(min(12, len(m)), random_state=1)
print("Catalán que PASA con D (al azar):")
display(pd.DataFrame({"texto (lado ca)": m.values,
                      "langdetect": [f"{det(t)[0]} ({det(t)[1]:.2f})" for t in m]}))

r = std.loc[[not acepta_D(t) for t in std["es"]], "es"].head(12)
print("Español que D SIGUE rechazando:")
display(pd.DataFrame({"texto (lado es)": r.values,
                      "langdetect": [f"{det(t)[0]} ({det(t)[1]:.2f})" for t in r]}))

Catalán que PASA con D (al azar):


,texto (lado ca),langdetect
0,Ets d'un camp de concentració?,ca (1.00)
1,"Perdona, m'he pensat que eres...",ca (1.00)
2,"condueix a baix, a la foscor!",ca (1.00)
3,"Treu-me d'aquí, va!",ca (1.00)
4,Ombres de fantasmes a terra.,ca (1.00)
5,- No en volen cap del 1r Escamot?,ca (0.86)
6,"Molt bé... de color, senyor.",ca (1.00)
7,- El cotxe va a petardejar.,ca (1.00)
8,Quant de temps vas estar a Bagdad?,ca (1.00)
9,No pot desitjar més desitjos.,ca (1.00)


Español que D SIGUE rechazando:


,texto (lado es),langdetect
0,No podemos quedarnos de brazos cruzados.,pt (1.00)
1,Porque no quiero decir algo que no debo.,pt (0.97)
2,"QUE TE HAGA LA CAMA, TE FRIEGUE EL BAÑO.",en (1.00)
3,¿queréis saber por qué le despidió realmente?,ca (1.00)
4,"La chica amish, Rebecca... ¿Te la tiraste?",it (1.00)
5,Es de suma importancia para nosotros conseguir...,pt (1.00)
6,Quiero saber qué le ocurrió a mi oficial.,ca (1.00)
7,"Esta cosa ve conspiradores, estafadores.",pt (1.00)
8,"Por favor, por favor, mira, mira, mira, mira.",pt (1.00)
9,"Hazme un favor, Jess, suelta ese cuchillo, anda.",it (1.00)


### Cómo leerlo
- **Sección 1:** si el español rechazado es bajo y el catalán que se cuela también, el filtro anda bien en el caso real. Si es alto, tira texto bueno.
- **Sección 2:** las preguntas son lo más corto del pipeline. Si acá el rechazo es alto, el filtro le saca justo los ejemplos de `¿` y `?` al dataset.
- **Sección 3:** muestra si lo que se pierde está concentrado en frases cortas (1-3 palabras). Eso sesgaría el dataset hacia frases largas.
- **Sección 5:** compara el filtro actual (A) con tres alternativas. Mejor variante = baja el "español rechazado" sin que suba mucho el "catalán que pasa". Las celdas de abajo muestran ejemplos para leer a ojo que no esté pasando basura.
